<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Read failures

**[Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/)** · Python for Practical AI Engineering · Module 5, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read a traceback from the bottom up, look at the values in a program when it gives a wrong result, and shrink a failure to a minimal example that shows its cause.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection: the setup installs httpx and downloads the data files. No account. |
| **You should know** | `count_tickets.py` with its source in `TICKETS_SOURCE`, from [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here. |
| **Tested** | Python 3.14.6 with httpx 0.28.1, in Jupyter on macOS, from a fresh start (*Restart and run all*). The line numbers in tracebacks can be different on your computer. |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m05/read-failures/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M05-L01-read-failures/read-failures-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1
import httpx; print("httpx", httpx.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, `tickets-excel.csv`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 14 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`), [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) (`remote.py` and `try_mock.py`), [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) (`config.py` and `.env.example`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
    "tickets-excel.csv": "cf0fbbce193bb1a595ce9207cc3f004aeedb118008e4ffa35585fda2f1167a95",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
from pathlib import Path

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> None:
    settings = load_settings()
    print(f"Source: {settings.source}")
    rows = load_rows(settings)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
    'ticket_cleaner/remote.py': '''\
import httpx


def fetch_rows(
    url: str, token: str | None = None, client: httpx.Client | None = None
) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
''',
    'try_mock.py': '''\
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()
''',
    'ticket_cleaner/config.py': '''\
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
    )
''',
    '.env.example': '''\
# Settings for ticket-cleaner. Set them as environment variables.
# TICKETS_SOURCE: a .csv or .json file, or "remote". Default: data/tickets.csv
TICKETS_SOURCE=data/tickets.csv
# TICKETS_URL: the web service for "remote". Default: the course's public file.
TICKETS_URL=
# TICKETS_API_TOKEN: secret. Never write a real token in this file.
TICKETS_API_TOKEN=
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `fresh("ticket_cleaner.parsing")` imports a module of the project again, so that the notebook uses the newest version of its file. (A plain `import` uses the copy that is already in memory: see [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/).)
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import importlib

def fresh(module_name):
    """Import a module of the project again, from its newest file."""
    for name in list(sys.modules):
        if name == "ticket_cleaner" or name.startswith("ticket_cleaner."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module(module_name)

def check_semicolon_fix():
    """Check fixed_rows: the semicolon file, read with the right separator."""
    expected_keys = ["id", "status", "category", "priority", "subject"]
    if not isinstance(fixed_rows, list) or len(fixed_rows) != 2:
        print("Not yet: fixed_rows must be a list with the 2 rows of the file.")
    elif list(fixed_rows[0]) != expected_keys:
        print(f"Not yet: the keys of the first row are {list(fixed_rows[0])}. Tell the reader which separator the file uses.")
    elif fixed_rows[0]["id"] != "T-4001":
        print("Not yet: the first row must be ticket T-4001.")
    else:
        print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A method, not luck

When a program fails, do not change things until it works. Follow a fixed order, as in the [decision tree on the lesson page](https://learning.nextia-ai.com/courses/python/m05/read-failures/#a-method-not-luck):

- **A traceback:** read the last line, then find the last frame in your own code.
- **A wrong result:** write down expected and actual, then find the first step where they differ.
- Then make the failure happen again on purpose, and shrink the input until only the cause is left: a **minimal failing example**. Fix it, and keep the example as a test.

## 2. Failure 1: a traceback

A teammate sends a JSON file from a different system. Run the cell: it saves it as `data/broken.json`. The second element is `null`.

In [ ]:
%%writefile data/broken.json
[
  {"id": "T-3001", "status": "open", "category": "billing", "priority": 2},
  null,
  {"id": "T-3003", "status": "closed", "category": "login", "priority": 3}
]

Run the program with it. `env={"TICKETS_SOURCE": "data/broken.json"}` is the same as `TICKETS_SOURCE=data/broken.json python count_tickets.py` in a terminal.

In [ ]:
run("count_tickets.py", env={"TICKETS_SOURCE": "data/broken.json"})

> **Check.** You should see `Source: data/broken.json`, then a traceback that ends with `AttributeError: 'NoneType' object has no attribute 'get'`, and `(exit code 1)`.

A **traceback** is the path of function calls that led to the error. Each call is a **frame**: a file, a line number, the function name, and the line of code. Read it **from the bottom up**:

| Read | What it tells you |
|---|---|
| The last line | The error type, `AttributeError`, and the message: a value of type `NoneType` has no `.get`. The only value of type `NoneType` is `None`. |
| The last frame | In `find_problems`, the line `raw.get(field)`. The `^^^^` marks show the exact part: `raw.get`. So `raw` is `None`. |
| The frame above it | `split_records` called `find_problems(raw)`, with one of the rows. |
| Higher frames | How the program got there: `main()` called `split_records`. |

`json.load` turns JSON `null` into `None`. So the cause is in the input: a record that is not a dictionary. Input from outside can contain anything, so the program must reject it with a reason. Run the cell: it adds the check at the start of `find_problems`, after the docstring.

In [ ]:
%%writefile ticket_cleaner/parsing.py
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    if not isinstance(raw, dict):
        return ["not a record"]
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected

In [ ]:
run("count_tickets.py", env={"TICKETS_SOURCE": "data/broken.json"})

> **Check.** You should see `Row 2 rejected: not a record` and `3 rows: 2 valid, 1 rejected.` `isinstance(raw, dict)` is `True` if `raw` is a dictionary, and the early `return` stops before any `raw.get`.

When the last frame is in a module that you did not write, such as `httpx` or `json`, go up to the last frame that is in **your** files.

## 3. Failure 2: a wrong result with no error

The help desk exports its tickets from a spreadsheet program. The setup downloaded that file as `data/tickets-excel.csv`. It looks the same as `tickets.csv`.

> **Predict.** Expected: 9 valid and 6 rejected, as for `tickets.csv`. Run the cell, and write down the actual result.

In [ ]:
run("count_tickets.py", env={"TICKETS_SOURCE": "data/tickets-excel.csv"})

> **Check.** You should see `Row 14 rejected: missing id; missing category`, `Row 15 rejected: missing id` and `15 rows: 0 valid, 15 rejected.` There is no traceback, but every row is "missing id", and you can see the IDs in the file.

**Find the first step where expected and actual differ.** `find_problems` says "missing id", so look at what it receives: the rows that `read_rows` returns. Run the cell.

In [ ]:
from pathlib import Path

files = fresh("ticket_cleaner.files")
rows = files.read_rows(Path("data/tickets-excel.csv"))
raw = rows[0]
print(raw)
print(list(raw))
print(raw.get("id"))

> **Check.** You should see `{'\ufeffid': 'T-1001', …}`, the keys `['\ufeffid', 'status', 'category', 'priority', 'subject']`, and `None`. The first key is not `'id'`: there is an invisible character, `\ufeff`, before `id`.

On your computer, the lesson does this with a **breakpoint**: add `breakpoint()` as the first line of `find_problems`, and run the program. It stops, and shows the debugger prompt `(Pdb)`. `p raw` prints a value, `n` runs the next line, `c` continues, and `q` quits. Remove the `breakpoint()` line afterwards. (In a notebook, `breakpoint()` in a cell opens the same debugger under the cell.)

**Make it small.** Is the problem the file, or the code? Make a two-line CSV file, written as a spreadsheet program writes it, with `encoding="utf-8-sig"`, and read it as `read_rows` does. Run the cell.

In [ ]:
import csv

with open("tiny.csv", "w", encoding="utf-8-sig") as file:
    file.write("id,status\nT-1,open\n")

with open("tiny.csv", encoding="utf-8", newline="") as file:
    print(list(csv.DictReader(file)))

> **Check.** You should see `[{'\ufeffid': 'T-1', 'status': 'open'}]`. Two lines of data, and the failure is still there. Spreadsheet programs put three bytes at the start of the file, to say "this file is UTF-8": a **byte order mark** (BOM). Reading with `encoding="utf-8"` keeps the mark as the character `\ufeff`, which becomes part of the first column name.

**Fix.** The encoding `utf-8-sig` reads UTF-8 and removes a byte order mark if there is one. A file without a mark reads the same as before. Run the cell: it changes the CSV line in `read_rows`, and deletes `tiny.csv`.

In [ ]:
%%writefile ticket_cleaner/files.py
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8-sig", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\n")
    os.replace(temporary, path)

In [ ]:
Path("tiny.csv").unlink(missing_ok=True)
run("count_tickets.py", env={"TICKETS_SOURCE": "data/tickets-excel.csv"})
run("count_tickets.py", env={"TICKETS_SOURCE": None})

> **Check.** Both runs end with `15 rows: 9 valid, 6 rejected.`: the spreadsheet file, and the default file without the variable.

## 4. Your turn: semicolons

In some countries, spreadsheet programs separate values with `;` instead of `,`. Run the cell: it saves `data/tickets-semicolon.csv`. Then run the program with it.

In [ ]:
%%writefile data/tickets-semicolon.csv
id;status;category;priority;subject
T-4001;open;billing;2;Wrong invoice total
T-4002;closed;login;3;Locked out

In [ ]:
run("count_tickets.py", env={"TICKETS_SOURCE": "data/tickets-semicolon.csv"})

> **Your turn.** Use the method:
>
> 1. Write down expected and actual. (Both rows are rejected.)
> 2. Find the first step where they differ: in a new cell, read the file with `files.read_rows`, and print `list(rows[0])`, as in section 3.
> 3. Write two sentences, in a Markdown cell or on paper: the cause, and what a fix would need to do.
> 4. Show the fix: the next cell reads the file into `fixed_rows`. Change it so that each row has the five columns. `csv.DictReader` has an argument for the separator. Then run the check cell.

In [ ]:
import csv

with open("data/tickets-semicolon.csv", encoding="utf-8-sig", newline="") as file:
    fixed_rows = list(csv.DictReader(file, delimiter=";"))

print(fixed_rows[0])

In [ ]:
check_semicolon_fix()

`csv.DictReader` splits on `,` by default, so it reads each line of this file as one column: `'id;status;category;priority;subject'`. For ticket-cleaner, you do not need to change `read_rows`: rejecting the file's rows with a clear reason is enough, and every row is already reported.

## 5. Failure case: changing code until the error goes away

After many small changes, the error is gone, but you do not know why. Some changes only hide the symptom. For example, `raw = raw or {}` for the `None` row. Run the cell: it shows what the `None` row would become.

In [ ]:
parsing = fresh("ticket_cleaner.parsing")
print(parsing.find_problems({}))
print(parsing.find_problems(None))

> **Check.** You should see `['missing id', 'missing status', 'missing category', 'missing priority']`, then `['not a record']`. With the guess, the real problem, a `null` in the file, is hidden behind four misleading problems. The `isinstance` check names it.

**Fix:** undo the guesses (`git diff` shows them, and `git restore` removes them). Collect one fact at a time. Change the code only when you can say the cause in one sentence.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m05/read-failures/). They count toward your certificate when you are signed in and enrolled. On your computer, do both investigations in your project, with `breakpoint()`, as in the guided practice of the lesson.

In this lesson you learned to read a traceback from the bottom up, to find the first step where expected and actual differ, to shrink a failure to a minimal failing example, and to reject a bad record with a reason instead of hiding it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m05/read-failures/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.